In [44]:
import json
import pickle
import random
from pathlib import Path

import numpy as np
import tensorflow as tf

from nltk.stem import PorterStemmer
from nltk.tokenize import wordpunct_tokenize

from sklearn.feature_extraction.text import CountVectorizer

from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Sequential

In [45]:
BASE_DIR = Path.cwd()

if BASE_DIR.name.lower() != "chatbot":
    BASE_DIR = BASE_DIR / "chatbot"

INTENTS_FILE = BASE_DIR / "intents.json"

with open(INTENTS_FILE, "r", encoding="utf-8") as file:
    intents_data = json.load(file)

print("File loaded:", INTENTS_FILE)
print("Total intents:", len(intents_data["intents"]))

File loaded: C:\Users\mugha\Desktop\E-Commerce-Product-Recommendation-System-main\chatbot\intents.json
Total intents: 21


In [46]:
texts = []
labels = []

for intent in intents_data["intents"]:
    for pattern in intent["patterns"]:
        texts.append(pattern)
        labels.append(intent["tag"])

classes = sorted(set(labels))

print("Total questions:", len(texts))
print("Total classes:", len(classes))
print("Classes:", classes)

Total questions: 1065
Total classes: 21
Classes: ['account', 'cancel_order', 'cart', 'checkout', 'contact_support', 'goodbye', 'greeting', 'help', 'offers', 'order_tracking', 'payment', 'product_comparison', 'product_details', 'product_search', 'recommendation', 'returns', 'shipping', 'similar_products', 'thanks', 'unknown', 'wishlist']


In [47]:
stemmer = PorterStemmer()


def clean_text(text):
    tokens = wordpunct_tokenize(text.lower())

    cleaned_tokens = []

    for token in tokens:
        if token.isalpha() and len(token) > 1:
            cleaned_tokens.append(stemmer.stem(token))

    return cleaned_tokens


vectorizer = CountVectorizer(
    tokenizer=clean_text,
    token_pattern=None,
    binary=True,
    lowercase=False
)

x_data = vectorizer.fit_transform(texts).toarray()
x_data = x_data.astype(np.float32)

words = vectorizer.get_feature_names_out().tolist()

y_numbers = []

for label in labels:
    y_numbers.append(classes.index(label))

y_data = np.array(y_numbers, dtype=np.int32)

print("Known words:", len(words))
print("Input shape:", x_data.shape)
print("Output shape:", y_data.shape)

Known words: 405
Input shape: (1065, 405)
Output shape: (1065,)


In [48]:
order = np.arange(len(x_data))

np.random.seed(42)
np.random.shuffle(order)

x_data = x_data[order]
y_data = y_data[order]

print("Data shuffled successfully.")

Data shuffled successfully.


In [49]:
tf.random.set_seed(42)

model = Sequential([
    Input(shape=(x_data.shape[1],)),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(len(classes), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                     │ (None, 64)                  │          25,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 21)                  │             693 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 28,757 (112.33 KB)

 Trainable params: 28,757 (112.33 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    x_data,
    y_data,
    epochs=200,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=0
)

In [51]:
best_training_accuracy = max(history.history["accuracy"])
best_validation_accuracy = max(history.history["val_accuracy"])

print(
    "Best training accuracy:",
    round(best_training_accuracy * 100, 2),
    "%"
)

print(
    "Best validation accuracy:",
    round(best_validation_accuracy * 100, 2),
    "%"
)

Best training accuracy: 99.41 %
Best validation accuracy: 85.92 %


In [52]:
model.save(BASE_DIR / "chatbot_model.keras")

with open(BASE_DIR / "words.pkl", "wb") as file:
    pickle.dump(words, file)

with open(BASE_DIR / "classes.pkl", "wb") as file:
    pickle.dump(classes, file)

print("Model and supporting files saved successfully.")

Model and supporting files saved successfully.


In [53]:
def predict_intent(message):
    message_numbers = vectorizer.transform([message]).toarray()
    message_numbers = message_numbers.astype(np.float32)

    probabilities = model.predict(
        message_numbers,
        verbose=0
    )[0]

    best_position = int(np.argmax(probabilities))

    predicted_tag = classes[best_position]
    confidence = float(probabilities[best_position])

    return predicted_tag, confidence

In [54]:
test_messages = [
    "Hey, how are you?",
    "Can you suggest something useful?",
    "I need information about a product",
    "Where has my package reached?",
    "I am ready to complete my purchase",
    "How do I get my money back?",
    "Can I talk to customer service?",
    "See you next time"
]

for message in test_messages:
    tag, confidence = predict_intent(message)

    print("Message:", message)
    print("Predicted intent:", tag)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("-" * 50)

Message: Hey, how are you?
Predicted intent: greeting
Confidence: 99.78 %
--------------------------------------------------
Message: Can you suggest something useful?
Predicted intent: recommendation
Confidence: 97.32 %
--------------------------------------------------
Message: I need information about a product
Predicted intent: product_details
Confidence: 97.49 %
--------------------------------------------------
Message: Where has my package reached?
Predicted intent: order_tracking
Confidence: 97.95 %
--------------------------------------------------
Message: I am ready to complete my purchase
Predicted intent: checkout
Confidence: 99.91 %
--------------------------------------------------
Message: How do I get my money back?
Predicted intent: returns
Confidence: 99.57 %
--------------------------------------------------
Message: Can I talk to customer service?
Predicted intent: contact_support
Confidence: 99.58 %
--------------------------------------------------
Message: See y